### Project Overview

This project demonstrates a Retrieval-Augmented Generation (RAG) chatbot using Groq for the Language Model (LLM) and local sentence embeddings for retrieval. The goal is to build a chatbot that can answer questions based on a provided dataset of patient reviews, ensuring responses are grounded in the given context.

**Dataset:**
The dataset used is `reviews.csv`, which contains patient review data. Each entry likely includes a review text along with other metadata such as review ID, visit ID, physician name, hospital name, and patient name. The chatbot will leverage these reviews to provide relevant answers.

**Model Used:**
*   **Embedding Model:** `all-MiniLM-L6-v2` from HuggingFace, a local sentence embedding model. This model is used to convert text (patient reviews and user questions) into numerical vector representations, allowing for semantic similarity search.
*   **Large Language Model (LLM):** `llama-3.1-8b-instant` via Groq API. Groq provides fast inference for large language models. The LLM is responsible for generating human-like responses based on the retrieved context.

**Workflow Strategies:**
1.  **Data Loading:** Patient reviews are loaded from `reviews.csv` using `CSVLoader` from LangChain. Each review is treated as a document.
2.  **Vector Database Creation:** A Chroma vector database is built using the loaded reviews. The `all-MiniLM-L6-v2` embedding model is used to generate embeddings for each review, which are then stored in Chroma. This process is batched to handle larger datasets efficiently.
3.  **Retrieval:** When a user asks a question, the question is also embedded using the same `all-MiniLM-L6-v2` model. A similarity search is performed against the Chroma vector database to find the most semantically relevant patient review chunks.
4.  **RAG Chain Setup:** A LangChain RAG chain is constructed:
    *   The `reviews_retriever` fetches relevant context from the vector database.
    *   A `ChatPromptTemplate` is used to structure the input to the LLM, combining a system prompt (guiding the LLM's behavior and context usage) and the user's question.
    *   The `ChatGroq` model (`llama-3.1-8b-instant`) processes the prompt and the retrieved context.
    *   `StrOutputParser` extracts the final string response from the LLM.
5.  **Chat Interface:** A Gradio web interface (`gr.ChatInterface`) is implemented to allow users to interact with the RAG chatbot in a conversational manner.

# Retrieval-Augmented Generation (RAG) Chatbot
### Using Groq (Free) + Local Sentence Embeddings

In [1]:
!pip install --quiet langchain langchain-core langchain-community langchain-chroma
!pip install --quiet sentence-transformers langchain-groq
!pip install --quiet gradio

In [2]:
import os
from google.colab import userdata

# Set your Groq API key (from console.groq.com — 100% Free)
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')  # store in Colab Secrets
# OR paste directly:
# os.environ['GROQ_API_KEY'] = 'gsk_your_key_here'

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_chroma import Chroma

REVIEWS_CSV_PATH = "/content/reviews.csv"
REVIEWS_CHROMA_PATH = "chroma_data"

print('Setup done!')

Setup done!


## Step 1 — Load the CSV Reviews

In [3]:
loader = CSVLoader(
    file_path=REVIEWS_CSV_PATH,
    source_column="review"
)
reviews = loader.load()
print(f'Loaded {len(reviews)} reviews')

Loaded 1005 reviews


## Step 2 — Build Vector Database (Local Embeddings — Completely Free, No Rate Limits)

In [4]:
# Local embedding model — no API key needed, no rate limits!
embedding_function = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"  # ~80MB, downloads once, runs locally
)

batch_size = 50
num_batches = (len(reviews) - 1) // batch_size + 1
reviews_vector_db = None

for i in range(0, len(reviews), batch_size):
    batch_docs = reviews[i:i + batch_size]
    current_batch_num = i // batch_size + 1
    print(f'Processing batch {current_batch_num}/{num_batches}...')

    if i == 0:
        reviews_vector_db = Chroma.from_documents(
            documents=batch_docs,
            embedding=embedding_function,
            persist_directory=REVIEWS_CHROMA_PATH
        )
    else:
        reviews_vector_db.add_documents(documents=batch_docs)

    print(f'Batch {current_batch_num} done.')

print('Vector database created successfully!')

/tmp/ipykernel_15647/1676232040.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Processing batch 1/21...
Batch 1 done.
Processing batch 2/21...
Batch 2 done.
Processing batch 3/21...
Batch 3 done.
Processing batch 4/21...
Batch 4 done.
Processing batch 5/21...
Batch 5 done.
Processing batch 6/21...
Batch 6 done.
Processing batch 7/21...
Batch 7 done.
Processing batch 8/21...
Batch 8 done.
Processing batch 9/21...
Batch 9 done.
Processing batch 10/21...
Batch 10 done.
Processing batch 11/21...
Batch 11 done.
Processing batch 12/21...
Batch 12 done.
Processing batch 13/21...
Batch 13 done.
Processing batch 14/21...
Batch 14 done.
Processing batch 15/21...
Batch 15 done.
Processing batch 16/21...
Batch 16 done.
Processing batch 17/21...
Batch 17 done.
Processing batch 18/21...
Batch 18 done.
Processing batch 19/21...
Batch 19 done.
Processing batch 20/21...
Batch 20 done.
Processing batch 21/21...
Batch 21 done.
Vector database created successfully!


### (Alternative) Load existing Chroma DB instead of rebuilding

In [5]:
# Run this cell INSTEAD of the one above if you already built the DB once
# embedding_function = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
# reviews_vector_db = Chroma(
#     persist_directory=REVIEWS_CHROMA_PATH,
#     embedding_function=embedding_function
# )
# print('Loaded existing vector DB')

## Step 3 — Test Retrieval

In [6]:
question = 'Has anyone complained about communication with the hospital staff?'
relevant_chunks = reviews_vector_db.similarity_search(question, k=3)

for i, chunk in enumerate(relevant_chunks):
    print(f'--- Chunk {i+1} ---')
    print(chunk.page_content)
    print()

--- Chunk 1 ---
review_id: 495
visit_id: 3777
review: The hospital staff was caring and attentive. The facilities were clean, and the medical care was excellent. However, the noise level in the ward was higher than expected, affecting my rest.
physician_name: Michael Watson
hospital_name: Vaughn PLC
patient_name: Mrs. Kelly Berry DVM

--- Chunk 2 ---
review_id: 813
visit_id: 1702
review: The hospital provided excellent care, and the staff was attentive to my needs. However, the noise level in the ward was a bit disruptive.
physician_name: David Carter
hospital_name: Jordan Inc
patient_name: Michael Herrera

--- Chunk 3 ---
review_id: 574
visit_id: 7162
review: The hospital's medical team was competent, but the noise level in the ward was disruptive, affecting my ability to rest.
physician_name: Teresa Frost
hospital_name: Walton LLC
patient_name: Kelsey Mills



## Step 4 — Set Up Groq LLM Chain (Free API)

In [7]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (
    PromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    ChatPromptTemplate,
)
from langchain_groq import ChatGroq

review_template_str = """Your job is to use patient reviews to answer questions about their experience at a hospital.
Use the following context to answer questions.
Be as detailed as possible, but don't make up any information that's not from the context.
If you don't know an answer, say you don't know.
When citing reviews, do NOT mention document IDs or technical identifiers.
Simply refer to them as 'One patient noted...', 'Another patient said...', or 'Several patients reported...'

{context}
"""

review_system_prompt = SystemMessagePromptTemplate(
    prompt=PromptTemplate(
        input_variables=["context"],
        template=review_template_str,
    )
)

review_human_prompt = HumanMessagePromptTemplate(
    prompt=PromptTemplate(
        input_variables=["question"],
        template="{question}",
    )
)

review_prompt_template = ChatPromptTemplate(
    input_variables=["context", "question"],
    messages=[review_system_prompt, review_human_prompt],
)

# --- Groq LLM (100% Free) ---
chat_model = ChatGroq(
    model="llama-3.1-8b-instant",  # Fast, free LLaMA 3.1 model via Groq
    temperature=0,
    groq_api_key=os.environ.get('GROQ_API_KEY')
)

reviews_retriever = reviews_vector_db.as_retriever(search_kwargs={"k": 10})

review_chain = (
    {"context": reviews_retriever, "question": RunnablePassthrough()}
    | review_prompt_template
    | chat_model
    | StrOutputParser()
)

print('Chain built! Ready to answer questions.')

Chain built! Ready to answer questions.


## Step 5 — Test the Full RAG Chain

In [8]:
question = 'Has anyone complained about communication with the hospital staff?'
response = review_chain.invoke(question)
print(response)

Yes, several patients reported issues with communication at the hospital. One patient noted that the hospital staff lacked proper communication among themselves, leading to confusion about their treatment plan.


## Step 6 — Launch Gradio Chat UI

In [ ]:
import gradio as gr

def respond_to_user_question(question: str, history: list) -> str:
    return review_chain.invoke(question)

interface = gr.ChatInterface(
    fn=respond_to_user_question,
    title="Hospital Review RAG Bot"
)

interface.launch(debug=True)